# 🚀 Transformations in Fivetran
## Quickstart vs dbt Transformations (With Examples)

---

# 📌 Overview

Fivetran supports two types of transformations:

1. Quickstart Transformations
2. dbt Transformations

---

# ⚡ 1. QUICKSTART TRANSFORMATIONS

## What it is
- Pre-built transformation models
- No coding required

## How it works
1. Load raw data
2. Enable Quickstart
3. Auto-create transformed tables

## Example

Input:
salesforce.opportunities

Output:
salesforce_opportunity_metrics

## Advantages
- Fast setup
- No SQL

## Limitations
- Less flexible
- Limited customization

---

# 🧠 2. DBT TRANSFORMATIONS

## What it is
- SQL-based transformation tool

## How it works
1. Load raw data
2. Write SQL models
3. Create final tables

## Example

Raw:
orders_raw

SQL:
SELECT customer_id, SUM(amount) AS total_spent
FROM orders_raw
GROUP BY customer_id

Output:
customer_spending

## Advantages
- Flexible
- Full control

## Limitations
- Requires SQL

---

# ⚖️ Comparison

Quickstart vs dbt:

- Setup: Easy vs Medium
- Coding: No vs Yes
- Flexibility: Low vs High

---

# 🔄 Flow

Source → Fivetran → Raw → Transform → Final → BI

---

# 🎯 Summary

Quickstart = ready models
dbt = custom SQL


# 🚀 No Primary Key in Source → How Fivetran Handles It (SQL Example)

---

# 📌 Problem Statement

What happens if the source table has NO primary key?

---

# 🧠 Why Primary Key is Important

Fivetran needs a primary key to:
- Identify unique rows
- Detect updates
- Avoid duplicates

---

# ❌ Without Primary Key

Fivetran cannot:
- Track updates properly
- Perform incremental sync reliably

---

# 🧪 Example (SQL Database)

## Step 1: Source Table (NO PRIMARY KEY)

CREATE TABLE orders (
    order_id INT,
    customer_name STRING,
    amount INT
);

No PRIMARY KEY

---

## Step 2: Insert Data

INSERT INTO orders VALUES (1, 'A', 100);
INSERT INTO orders VALUES (1, 'A', 100);
INSERT INTO orders VALUES (2, 'B', 200);

---

# 🔥 Problem

Duplicate rows exist
Fivetran cannot identify unique rows

---

# ⚙️ What Fivetran Does

## Case 1: No PK + No Incremental Column

→ Uses FULL TABLE SYNC
→ Reloads entire table every time

Issues:
- High MAR cost
- Poor performance
- Duplicate data

---

## Case 2: No PK → Fivetran creates _fivetran_id

Result:

order_id | customer_name | amount | _fivetran_id
1 | A | 100 | abc123
1 | A | 100 | xyz456
2 | B | 200 | pqr789

Duplicates still exist

---

# 🔄 Update Scenario

UPDATE orders SET amount = 150 WHERE order_id = 1;

Result:

order_id | amount | _fivetran_id
1 | 100 | abc123
1 | 150 | def456

Duplicates + inconsistency

---

# ⚠️ Key Issues

- Duplicate records
- Incorrect updates
- Full reload
- High MAR

---

# 🚀 Solutions

## Add Primary Key
ALTER TABLE orders ADD PRIMARY KEY (order_id);

## Composite Key
PRIMARY KEY (order_id, customer_name);

## Add Incremental Column
updated_at TIMESTAMP

## Use dbt Deduplication

SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _fivetran_synced DESC) AS rn
    FROM orders
)
WHERE rn = 1;

---

# 🧠 Summary

No PK → full sync + duplicates
With PK → proper incremental sync


# 🚀 Table Sync Mode vs Soft Delete Mode vs History Mode in Fivetran

---

# 📌 Overview

Fivetran provides multiple strategies to handle updates & deletes:

1. Table Sync Mode (Hard Replace)
2. Soft Delete Mode
3. History Mode (SCD Type 2)

---

# 🧠 1. TABLE SYNC MODE

## What it is
Entire table is replaced during each sync

## How it works
1. Extract full data
2. Truncate destination
3. Reload table

## Example

Before:
id | name
1  | A
2  | B

After delete:
id | name
1  | A

Destination:
id | name
1  | A

## Pros
- Clean mirror
- Simple

## Cons
- No history
- Expensive

---

# 🧩 2. SOFT DELETE MODE

## What it is
Rows are marked instead of deleted

## How it works
- _fivetran_deleted = TRUE

## Example

id | name | _fivetran_deleted
1  | A    | FALSE
2  | B    | TRUE

## Pros
- Keeps deleted data

## Cons
- No full history
- Needs filtering

---

# 🧠 3. HISTORY MODE

## What it is
Tracks full history (SCD Type 2)

## How it works
- Insert new row on change
- Keep old row

## Example

Initial:
id | name | start | end
1  | A    | t1    | NULL

After update:
id | name | start | end
1  | A    | t1    | t2
1  | B    | t2    | NULL

## Columns
- _fivetran_start
- _fivetran_end

## Pros
- Full history
- Best for analytics

## Cons
- More storage
- Complex queries

---

# ⚖️ Comparison

Table Sync → No history
Soft Delete → Deleted flag
History Mode → Full history

---

# 🎯 Summary

Table Sync = replace
Soft Delete = flag
History Mode = version tracking


# 🚀 Hashed vs Unhashed Data in Fivetran (With Examples)

---

# 📌 Overview

Fivetran handles sensitive data using hashing:

1. Unhashed (Plain Data)
2. Hashed (Secure Data)

---

# 🧠 1. UNHASHED DATA

## What it is
Data stored in original readable form

## Example

email | name
user@gmail.com | A
test@yahoo.com | B

## Advantages
- Readable
- Easy analytics

## Risks
- Security issues
- Exposes PII

---

# 🔐 2. HASHED DATA

## What it is
Data converted into fixed hash string

## Example

email (hashed)
a8f5f167f44f...
b6d81b360a56...

## Properties
- One-way
- Same input → same hash

## Advantages
- Secure
- GDPR compliant

## Limitations
- Not readable
- Cannot reverse

---

# 🔄 Example

Original:
user@gmail.com

Hashed:
a8f5f167f44f4964e6c998dee827110c

---

# ⚖️ Comparison

Unhashed → readable
Hashed → secure

---

# 🎯 Summary

Unhashed = plain data
Hashed = protected data


# 🚀 Schema Change Settings in Fivetran

---

# 📌 Overview

Fivetran automatically handles schema changes in source systems.

Schema change = change in:
- Columns
- Data types
- Tables

---

# 🧠 Why Important

- Source systems evolve
- New columns added
- Old columns removed
- Data types change

Fivetran ensures no pipeline break and auto adapts

---

# 🔄 Types of Schema Changes

## 1. New Column Added

Before:
id | name

After:
id | name | age

Fivetran:
- Adds new column automatically

---

## 2. Column Removed

Fivetran:
- Keeps column in destination
- Stops updating it

---

## 3. Data Type Change

Example:
amount INT → FLOAT

Fivetran:
- Promotes type

---

## 4. Table Added

- New table auto created

---

## 5. Table Removed

- Table not deleted automatically

---

# ⚙️ Settings in UI

- Enable/disable tables
- Enable/disable columns
- Block columns
- Choose sync mode

---

# 🧠 Behavior

- Detect → Compare → Update
- No breaking changes
- Keeps old data

---

# ⚠️ Important Rules

- Rename = new column
- Old columns not auto removed
- No type downgrade

---

# 🚀 Example

Initial:
id | amount
1  | 100

After change:
1  | 100.5

Result:
amount → FLOAT

---

# ⚠️ Issues

- Duplicate columns (rename)
- Unexpected STRING
- Old columns remain

---

# 🚀 Best Practices

- Monitor schema
- Clean unused columns
- Avoid renaming
- Use dbt

---

# 🎯 Summary

Fivetran auto handles schema changes safely without breaking pipelines.
